# 04. Governed Incident Blackboard

This notebook is the product-facing exemplar for AETHER.

We will model one shared incident board where:

- observations land on the board
- candidate remediation actions share the same history
- AETHER derives which action is truly ready
- lease state tells us who may act now
- stale attempts are fenced after handoff
- one tuple explanation shows why the current answer is valid


In [ ]:
from pathlib import Path
import subprocess
import sys
from textwrap import dedent

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next(
    (
        path
        for path in candidate_roots
        if (path / "python").exists() and (path / "Cargo.toml").exists()
    ),
    None,
)

if repo_root is None:
    repo_root = Path("/content/AETHER")
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/fyremael/AETHER", str(repo_root)],
            check=True,
        )

python_root = repo_root / "python"
if str(python_root) not in sys.path:
    sys.path.insert(0, str(python_root))

repo_root


In [ ]:
from notebooks.colab_setup import ensure_rust_toolchain, pretty_json, start_http_service, stop_http_service
from aether_sdk import AetherClient, make_datom, value_entity, value_string, value_u64

ensure_rust_toolchain()
session = start_http_service(repo_root)
client = AetherClient(session.base_url)

pretty_json(client.health())


## Seed The Incident Board History

These datoms mirror the core board story from the governed incident blackboard demo.


In [ ]:
board_history = [
    make_datom(entity=1, attribute=1, value=value_string("latency_spike"), element=1, op="Add"),
    make_datom(entity=1, attribute=1, value=value_string("saturation_alert"), element=2, op="Add"),
    make_datom(entity=202, attribute=2, value=value_string("drain-read-replica"), element=3),
    make_datom(entity=202, attribute=8, value=value_string("done"), element=4),
    make_datom(entity=201, attribute=2, value=value_string("shift-read-traffic"), element=5),
    make_datom(entity=201, attribute=3, value=value_entity(1), element=6),
    make_datom(entity=201, attribute=4, value=value_entity(202), element=7, op="Add"),
    make_datom(entity=201, attribute=5, value=value_string("latency_spike"), element=8, op="Add"),
    make_datom(entity=201, attribute=6, value=value_string("approved"), element=9),
    make_datom(entity=201, attribute=7, value=value_string("clear"), element=10),
    make_datom(entity=203, attribute=2, value=value_string("restart-primary"), element=11),
    make_datom(entity=203, attribute=3, value=value_entity(1), element=12),
    make_datom(entity=203, attribute=5, value=value_string("saturation_alert"), element=13, op="Add"),
    make_datom(entity=203, attribute=6, value=value_string("approved"), element=14),
    make_datom(entity=203, attribute=7, value=value_string("suppressed"), element=15),
    make_datom(entity=201, attribute=9, value=value_string("remediator-a"), element=16),
    make_datom(entity=201, attribute=10, value=value_u64(1), element=17),
    make_datom(entity=201, attribute=11, value=value_string("active"), element=18),
    make_datom(entity=201, attribute=9, value=value_string("remediator-b"), element=19),
    make_datom(entity=201, attribute=10, value=value_u64(2), element=20),
]

client.append(board_history)
pretty_json(client.history())


In [ ]:
board_document = dedent(
    """
    schema v1 {
      attr incident.observation: SetAddWins<String>
      attr action.title: ScalarLWW<String>
      attr action.incident: RefScalar<Entity>
      attr action.depends_on: RefSet<Entity>
      attr action.requires_signal: SetAddWins<String>
      attr action.approval_state: ScalarLWW<String>
      attr action.suppression_state: ScalarLWW<String>
      attr action.status: ScalarLWW<String>
      attr action.claimed_by: ScalarLWW<String>
      attr action.lease_epoch: ScalarLWW<U64>
      attr action.lease_state: ScalarLWW<String>
    }

    predicates {
      candidate_action(Entity)
      execution_attempt(Entity, String, U64)
      incident_observation(Entity, String)
      action_title(Entity, String)
      action_incident(Entity, Entity)
      action_depends_on(Entity, Entity)
      action_requires_signal(Entity, String)
      action_approval_state(Entity, String)
      action_suppression_state(Entity, String)
      action_status(Entity, String)
      action_claimed_by(Entity, String)
      action_lease_epoch(Entity, U64)
      action_lease_state(Entity, String)
      observation_active(Entity, String)
      dependency_closure(Entity, Entity)
      action_complete(Entity)
      dependency_blocked(Entity)
      action_missing_signal(Entity)
      action_suppressed(Entity)
      action_approved(Entity)
      lease_active(Entity, String, U64)
      active_claim(Entity)
      action_ready(Entity)
      board_action(Entity, String, String, String)
      ready_action_detail(Entity, String)
      execution_authorized(Entity, String, U64)
      execution_authorized_detail(Entity, String, String, U64)
      execution_rejected_stale(Entity, String, U64)
      execution_rejected_stale_detail(Entity, String, String, U64)
    }

    facts {
      candidate_action(entity(201))
      candidate_action(entity(203))
      execution_attempt(entity(201), "remediator-a", 1)
      execution_attempt(entity(201), "remediator-b", 1)
      execution_attempt(entity(201), "remediator-a", 2)
      execution_attempt(entity(201), "remediator-b", 2)
    }

    rules {
      observation_active(incident, signal) <- incident_observation(incident, signal)
      dependency_closure(action, dep) <- action_depends_on(action, dep)
      dependency_closure(action, dep) <- action_depends_on(action, mid), dependency_closure(mid, dep)
      action_complete(action) <- action_status(action, "done")
      dependency_blocked(action) <- dependency_closure(action, dep), not action_complete(dep)
      action_missing_signal(action) <- action_requires_signal(action, signal), action_incident(action, incident), not incident_observation(incident, signal)
      action_suppressed(action) <- action_suppression_state(action, "suppressed")
      action_approved(action) <- action_approval_state(action, "approved")
      lease_active(action, worker, epoch) <- action_claimed_by(action, worker), action_lease_epoch(action, epoch), action_lease_state(action, "active")
      active_claim(action) <- lease_active(action, worker, epoch)
      action_ready(action) <- candidate_action(action), action_approved(action), not dependency_blocked(action), not action_missing_signal(action), not action_suppressed(action), not active_claim(action), not action_complete(action)
      board_action(action, title, approval, suppression) <- candidate_action(action), action_title(action, title), action_approval_state(action, approval), action_suppression_state(action, suppression)
      ready_action_detail(action, title) <- action_ready(action), action_title(action, title)
      execution_authorized(action, worker, epoch) <- execution_attempt(action, worker, epoch), lease_active(action, worker, epoch)
      execution_authorized_detail(action, title, worker, epoch) <- execution_authorized(action, worker, epoch), action_title(action, title)
      execution_rejected_stale(action, worker, epoch) <- execution_attempt(action, worker, epoch), not lease_active(action, worker, epoch)
      execution_rejected_stale_detail(action, title, worker, epoch) <- execution_rejected_stale(action, worker, epoch), action_title(action, title)
    }

    materialize {
      observation_active
      dependency_closure
      dependency_blocked
      action_missing_signal
      action_suppressed
      action_ready
      board_action
      ready_action_detail
      execution_authorized
      execution_authorized_detail
      execution_rejected_stale
      execution_rejected_stale_detail
    }

    query observations_current {
      current
      goal observation_active(incident, signal)
      keep incident, signal
    }

    query board_actions_current {
      current
      goal board_action(action, title, approval, suppression)
      keep action, title, approval, suppression
    }

    query ready_at_e15 {
      as_of e15
      goal ready_action_detail(action, title)
      keep action, title
    }

    query authorized_at_e18 {
      as_of e18
      goal execution_authorized_detail(action, title, worker, epoch)
      keep action, title, worker, epoch
    }

    query authorized_current {
      current
      goal execution_authorized_detail(action, title, worker, epoch)
      keep action, title, worker, epoch
    }

    query stale_current {
      current
      goal execution_rejected_stale_detail(action, title, worker, epoch)
      keep action, title, worker, epoch
    }
    """
)

board_response = client.run_document(board_document)
pretty_json(board_response["queries"])


In [ ]:
for query_name in [
    "observations_current",
    "board_actions_current",
    "ready_at_e15",
    "authorized_at_e18",
    "authorized_current",
    "stale_current",
]:
    rows = client.run_named_query(board_document, query_name=query_name)["rows"]
    print(query_name)
    print([row["values"] for row in rows])
    print()


In [ ]:
authorized_current = client.run_named_query(board_document, query_name="authorized_current")
tuple_id = authorized_current["rows"][0]["tuple_id"]
trace = client.explain_tuple(tuple_id)

print(f"Explaining tuple_id={tuple_id}")
pretty_json(trace)


## Why This Notebook Matters

This is the adjacent-next product story in runnable form.

The board answers five operator questions directly:

- what is active now
- which action is truly ready
- who may act now
- what changed since the last handoff
- why the answer is true


In [ ]:
# Uncomment this cell when you are done with the notebook.
# stop_http_service(session)
